# Crash Course — 신경신호 시간-주파수 분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/neural-time-frequency.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/neural-time-frequency.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 예시 1 — wavelet 한 번으로 dB power와 ITPC를 함께

In [ ]:
import numpy as np

rng = np.random.default_rng(20)
fs = 250
t = np.arange(-0.8, 1.2, 1 / fs)
n_tr = 50
# 가상 EEG 시행: 배경 잡음 + 자극 후 0–0.4초 위상 고정 5 Hz + 0.4–0.9초 alpha(10 Hz) 감소
X = rng.normal(0, 1, (n_tr, len(t)))
for k in range(n_tr):
    X[k] += (1 - 0.6 * ((t > 0.4) & (t < 0.9))) * np.sin(2 * np.pi * 10 * t + rng.uniform(0, 2 * np.pi))
    X[k] += 1.0 * ((t > 0) & (t < 0.4)) * np.sin(2 * np.pi * 5 * t)

def morlet_tf(X, freqs, cycles, fs):
    """시행 × 시간 -> (주파수 × 시행 × 시간) 복소 계수. FFT 곱 + 날개 자르기 + 주파수 영역 정규화"""
    wt = np.arange(-2, 2 + 1 / fs, 1 / fs); half = (len(wt) - 1) // 2
    L = X.shape[1] + len(wt) - 1
    FX = np.fft.fft(X, L, axis=1)
    out = np.empty((len(freqs), *X.shape), dtype=complex)
    for i, (f, c) in enumerate(zip(freqs, cycles)):
        s = c / (2 * np.pi * f)
        W = np.fft.fft(np.exp(2j * np.pi * f * wt - wt ** 2 / (2 * s ** 2)), L)
        out[i] = np.fft.ifft(FX * (W / np.abs(W).max()), axis=1)[:, half:half + X.shape[1]]
    return out

freqs = np.arange(3, 31, 1.0)
C = morlet_tf(X, freqs, np.linspace(3, 8, len(freqs)), fs)
power = (np.abs(C) ** 2).mean(axis=1)                       # total power (주파수 × 시간)
itpc = np.abs(np.exp(1j * np.angle(C)).mean(axis=1))        # 시행 간 위상 일관성
base = (t > -0.6) & (t < -0.2)
db = 10 * np.log10(power / power[:, base].mean(axis=1, keepdims=True))

def at(M, f, a, b):
    return M[np.argmin(np.abs(freqs - f)), (t > a) & (t < b)].mean()
print("10 Hz, 0.5–0.8초: %+.1f dB | ITPC %.2f" % (at(db, 10, 0.5, 0.8), at(itpc, 10, 0.5, 0.8)))
print(" 5 Hz, 0.1–0.3초: %+.1f dB | ITPC %.2f" % (at(db, 5, 0.1, 0.3), at(itpc, 5, 0.1, 0.3)))
print("기저선 ITPC 평균 %.2f (시행 50개에서 무작위 위상의 기대치 약 %.2f)" % (itpc[:, base].mean(), np.sqrt(np.pi / 4 / n_tr)))

### 예시 2 — 시간 경과에 대한 cluster 순열 검정

In [ ]:
import numpy as np
from scipy import ndimage, stats

rng = np.random.default_rng(21)
t = np.linspace(-0.5, 1.0, 151)                  # 10 ms 간격의 시간축
n = 25                                           # 조건마다 25시행 (또는 25명)
def courses(effect):
    """시행별 alpha 파워 시간 경과(dB). 이웃 시점끼리 닮은 잡음 + (선택) 0.3–0.7초 감소"""
    noise = ndimage.gaussian_filter1d(rng.normal(0, 1, (n, len(t))), 4, axis=1) * 3
    return noise - (1.2 * ((t > 0.3) & (t < 0.7)) if effect else 0)

A, B = courses(False), courses(True)
thr = stats.t.ppf(0.975, 2 * n - 2)

def max_cluster_mass(tv):
    lab, k = ndimage.label(np.abs(tv) > thr)
    return lab, [np.abs(tv[lab == i]).sum() for i in range(1, k + 1)]

t_obs = stats.ttest_ind(B, A, axis=0).statistic
lab, mass = max_cluster_mass(t_obs)
both = np.vstack([A, B]); g = np.r_[np.zeros(n), np.ones(n)]
null = []
for _ in range(1000):
    p = rng.permutation(g)
    _, m = max_cluster_mass(stats.ttest_ind(both[p == 1], both[p == 0], axis=0).statistic)
    null.append(max(m) if m else 0)
for i, m in enumerate(mass, 1):
    p_c = (np.sum(np.array(null) >= m) + 1) / (len(null) + 1)
    seg = t[lab == i]
    print("cluster %d: %.2f–%.2f초, mass %.1f, 순열 p = %.3f" % (i, seg[0], seg[-1], m, p_c))